In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
# Create SparkSession
# MVN repo to load JAR of spark-kafka connect
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("KafkaStream") \
    .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1") \
    .config("spark.executor.memory", "1g") \
    .getOrCreate()

spark

:: loading settings :: url = jar:file:/opt/bitnami/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /opt/bitnami/spark/.ivy2.5.2/cache
The jars for the packages stored in: /opt/bitnami/spark/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f4a86fe5-f898-4827-806e-e279216eaf94;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#sca

26/04/22 14:03:59 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 1239810 ms exceeds timeout 120000 ms
26/04/22 14:04:00 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 1245135 ms exceeds timeout 120000 ms
26/04/22 14:04:00 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/04/22 14:04:00 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/04/22 14:04:00 WARN StandaloneSchedulerBackend: Executor to kill 1 does not exist!
26/04/22 14:04:00 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:748)
	at org.apache.spark.storage.BlockManagerMasterE

In [3]:
# Create SparkSession ,  Normal. spark without kafka Jar
spark =  SparkSession.builder \
                    .master("spark://spark-master:7077") \
                    .appName("example") \
                    .config("spark.executor.memory", "2g") \
                    .getOrCreate()
# spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 14:14:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import time
from pyspark.sql.functions import to_json, struct, col

In [4]:
schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

# Triggers

In [5]:
df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

In [3]:
# Note, you can identify whether a DataFrame/Dataset has streaming data or not by using df.isStreaming.
df.isStreaming()

NameError: name 'df' is not defined

In [6]:
processed = df.filter(df["LocationID"] > 10)

In [7]:
#to convert dataframe into json key value pair 

kafka_df = processed.select(
    to_json(struct("*")).alias("value")
)

#### ProcessingTime : every 10 seconds, check for new data and process whatever is available

In [ ]:
kafka_df.writeStream \
    .format("kafka") \
    .option("checkpointLocation", "/app/checkpoint/") \
    .option("topic", "spark_stream_topic") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .start() \
    .awaitTermination()

26/04/13 14:53:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


#### Once : Runs ONE micro-batch, processes all currently available data, then stops automatically

In [11]:
# check point location should be different keep this in record not 2 query hit the same check point
kafka_df.writeStream \
    .format("kafka") \
    .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/") \
    .option("topic", "spark_stream_topic") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .outputMode("append") \
    .trigger(once=True) \
    .start() \
    .awaitTermination()

26/04/01 11:51:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


#### availableNow : process all data available and stop but it make mmultiple batches off that data instead of processing all in one batch 

In [9]:
# All data → MULTIPLE batches → STOP
# Unlike once=True, it does NOT force everything into a single batch.
kafka_df.writeStream \
    .format("kafka") \
    .option("checkpointLocation", "/app/checkpoint2/") \
    .option("topic", "spark_stream_topic") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start() \
    .awaitTermination()

26/03/30 14:07:41 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


 #### Continuous : Runs the query in continuous processing mode (near real-time), instead of micro-batches.

In [ ]:
# Limitations of continuous mode
# 1. Very limited operations supported

# You CANNOT use:

# groupBy
# join
# aggregation
# window functions
# many transformations

In [10]:
from pyspark.sql.functions import col, from_json

# Read stream from kafka
df = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers",  "broker:9092") \
  .option("subscribe", "spark_stream_topic") \
  .load()

json_df = df.selectExpr("CAST(value AS STRING) as json_value")

parsed_df = json_df.select(
    from_json(col("json_value"), schema).alias("data")
)

final_df = parsed_df.select("data.*")

final_df = final_df.filter(final_df["LocationID"] > 100)

In [ ]:
query = final_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("numRows", 10) \
    .option("truncate", False) \
    .start()

query.awaitTermination()

26/03/31 07:16:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b1e9e171-7bcc-4245-a9d7-386bef98527e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/31 07:16:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----------+-------+----+------------+
|LocationID|Borough|Zone|service_zone|
+----------+-------+----+------------+
+----------+-------+----+------------+



In [11]:
kafka_df = final_df.select(
    to_json(struct("*")).alias("value")
)

query=kafka_df.writeStream \
    .format("kafka") \
    .option("startingOffsets", "earliest") \
    .option("checkpointLocation", "/app/checkpoint2/") \
    .option("topic", "spark_continuous_stream_topic") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .outputMode("append") \
    .trigger(continuous='5 seconds') \
    .start()
query.awaitTermination(timeout=60)  
query.stop()  

26/03/31 07:40:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/31 07:40:52 WARN ContinuousExecution: Disabling AQE since the query runs with continuous mode.
26/03/31 07:41:52 WARN TaskSetManager: Lost task 0.0 in stage 1.0 (TID 4) (172.18.0.5 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 1 cancelled Continuous execution finished for query [id = 490127ce-41a3-43ec-ac6a-eb0fcfa9626b, runId = 27b1817c-55af-4eab-872d-3552c5fa7b91] SQLSTATE: XXKDA)
26/03/31 07:41:52 WARN TaskSetManager: Lost task 2.0 in stage 1.0 (TID 6) (172.18.0.5 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 1 cancelled Continuous execution finished for query [id = 490127ce-41a3-43ec-ac6a-eb0fcfa9626b, runId = 27b1817c-55af-4eab-872d-3552c5fa7b91] SQLSTATE: XXKDA)
26/03/31 07:41:52 WARN TaskSetManager: Lost task 1.0 in stage 1.0 (TID 5) (172.18.0.2 executor 1): TaskKilled (Stage cancelled: [

# Fetch data from kafka

In [13]:
#.option("startingOffsets", "earliest") \ #The issue is your Kafka startingOffsets. By default, streaming reads from latest — meaning it only picks up new messages arriving after the stream starts. Since your data is already in Kafka, it sees nothing new.

from pyspark.sql.functions import col, from_json
import time

df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .option("subscribe", "spark_stream_topic") \
    .option("startingOffsets", "earliest") \
    .load()

json_df = df.selectExpr("CAST(value AS STRING) as json_value")

parsed_df = json_df.select(
    from_json(col("json_value"), schema).alias("data")
)

final_df = parsed_df.select("data.*")

query = final_df.writeStream \
    .format("csv") \
    .option("path", "/app/stream_output/") \
    .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/") \
    .option("header", "true") \
    .trigger(once=True) \
    .start()

query.awaitTermination()

26/04/01 11:56:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/01 12:21:27 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 1379922 ms exceeds timeout 120000 ms
26/04/01 12:21:28 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 1374865 ms exceeds timeout 120000 ms
26/04/01 12:21:28 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/04/01 12:21:28 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.3: worker lost: Not receiving heartbeat for 60 seconds
26/04/01 12:21:28 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.s

# How Foreachbatch Works

In [14]:
import time

from pyspark.sql import SparkSession, DataFrame

df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .option("subscribe", "spark_stream_topic") \
    .option("startingOffsets", "earliest") \
    .load()

json_df = df.selectExpr("CAST(value AS STRING) as json_value")

parsed_df = json_df.select(
    from_json(col("json_value"), schema).alias("data")
)

final_df = parsed_df.select("data.*")

def write_to_multiple_sinks(batch_df, batch_id):
    batch_df.write.format("parquet").save("/app/output/foreachbatch-table1")
    batch_df.write.format("parquet").save("/app/output/foreachbatch-table2")
    batch_df.write.format("parquet").save("/app/output/foreachbatch-table3")

query = final_df\
        .writeStream\
        .foreachBatch(write_to_multiple_sinks)\
        .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/")\
        .trigger(once=True)\
        .start()

26/03/31 09:27:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/31 10:16:00 WARN HeartbeatReceiver: Removing executor 6 with no recent heartbeats: 1005717 ms exceeds timeout 120000 ms
26/03/31 10:16:00 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.2: worker lost: Not receiving heartbeat for 60 seconds
26/03/31 10:16:00 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/03/31 10:16:00 WARN HeartbeatReceiver: Removing executor 7 with no recent heartbeats: 1002190 ms exceeds timeout 120000 ms
26/03/31 10:16:00 WARN StandaloneSchedulerBackend: Executor to kill 6 does not exist!
26/03/31 10:16:00 WARN StandaloneSchedulerBackend: Executor to kill 7 does not exist!


# Windowing in streaming

In [7]:
# Tubling Window
from pyspark.sql.functions import col, from_json
import time
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import window, count

df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

processed = df.withColumn("event_time", current_timestamp())

windowed_df = processed \
    .groupBy(window("event_time", "1 minutes"),"Borough") \
    .agg(count("*").alias("record_count"))



# Write results to console
query = windowed_df.writeStream \
    .trigger(processingTime="5 seconds") \
    .outputMode("complete") \
    .format("console") \
    .start()

# query = windowed_df.writeStream \
#     .format("csv") \
#     .outputMode("complete") \
#     .option("path", "/app/stream_output/") \
#     .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/") \
#     .option("header", "true") \
#     .trigger(processingTime="5 seconds") \
#     .start()

# query.awaitTermination() Start the stream and keep this thread blocked until the stream stops , like start in attached mode

# == Physical Plan ==
# WriteToDataSourceV2 (10)
# +- * HashAggregate (9)
#    +- StateStoreSave (8)
#       +- * HashAggregate (7)
#          +- StateStoreRestore (6)
#             +- * HashAggregate (5)
#                +- Exchange (4)
#                   +- * HashAggregate (3)
#                      +- * Project (2)
#                         +- Scan csv  (1)

26/04/02 09:31:27 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-85e8d93c-74e7-456c-9763-fade70d1f471. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/02 09:31:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/02 09:31:27 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/04/02 09:31:44 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 9034 milliseconds


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+-------------+------------+
|              window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:31...|       Queens|         138|
|{2026-04-02 09:31...|    Manhattan|         138|
|{2026-04-02 09:31...|Staten Island|          40|
|{2026-04-02 09:31...|          EWR|           2|
|{2026-04-02 09:31...|     Brooklyn|         122|
|{2026-04-02 09:31...|        Bronx|          86|
|{2026-04-02 09:31...|      Unknown|           2|
|{2026-04-02 09:31...|          N/A|           2|
+--------------------+-------------+------------+



26/04/02 09:31:56 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6613 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-------------+------------+
|              window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:31...|       Queens|         276|
|{2026-04-02 09:31...|    Manhattan|         276|
|{2026-04-02 09:31...|Staten Island|          80|
|{2026-04-02 09:31...|          EWR|           4|
|{2026-04-02 09:31...|     Brooklyn|         244|
|{2026-04-02 09:31...|        Bronx|         172|
|{2026-04-02 09:31...|      Unknown|           4|
|{2026-04-02 09:31...|          N/A|           4|
+--------------------+-------------+------------+



26/04/02 09:33:25 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5825 milliseconds


-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+-------------+------------+
|              window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:31...|       Queens|         276|
|{2026-04-02 09:33...|       Queens|         138|
|{2026-04-02 09:33...|Staten Island|          40|
|{2026-04-02 09:31...|    Manhattan|         276|
|{2026-04-02 09:31...|Staten Island|          80|
|{2026-04-02 09:33...|      Unknown|           2|
|{2026-04-02 09:33...|     Brooklyn|         122|
|{2026-04-02 09:33...|    Manhattan|         138|
|{2026-04-02 09:31...|          EWR|           4|
|{2026-04-02 09:33...|        Bronx|          86|
|{2026-04-02 09:31...|     Brooklyn|         244|
|{2026-04-02 09:31...|        Bronx|         172|
|{2026-04-02 09:31...|      Unknown|           4|
|{2026-04-02 09:33...|          EWR|           2|
|{2026-04-02 09:31...|          N/A|           4|
|{2

In [13]:
query.stop()

26/04/03 15:21:35 WARN DAGScheduler: Failed to cancel job group 92fc50c1-b089-462a-89b7-356ea1d4a70e. Cannot find active jobs for it.
26/04/03 15:21:35 WARN DAGScheduler: Failed to cancel job group 92fc50c1-b089-462a-89b7-356ea1d4a70e. Cannot find active jobs for it.


In [9]:
# sliding Window
from pyspark.sql.functions import col, from_json
import time
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import window, count

df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

processed = df.withColumn("event_time", current_timestamp())

# window(col("timestamp"), "10 minutes", "5 minutes") Window size: 10 mins, Slide: 5 mins 
# .withWatermark("event_time", "2 minutes") “I may receive late data, but only up to 2 minutes late. After that, ignore it and close state.”
# Why watermark becomes important: it tells spark “After this delay, no more late data is expected. You can safely delete old window states. and becasue it is overallping its states 
# records will increase fastly so to free fro OOM we use watermark”
windowed_df = processed \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(window(col("event_time"), "1 minutes", "30 seconds") ,"Borough") \
    .agg(count("*").alias("record_count"))



query = windowed_df.writeStream \
    .format("console") \
    .outputMode("complete") \
    .trigger(processingTime="5 seconds") \
    .start()

# query.awaitTermination()


# it generated sub execution IDs at SQL/DataFrame tab
# == Physical Plan ==
# WriteToDataSourceV2 (12)
# +- * HashAggregate (11)
#    +- StateStoreSave (10)
#       +- * HashAggregate (9)
#          +- StateStoreRestore (8)
#             +- * HashAggregate (7)
#                +- Exchange (6)
#                   +- * HashAggregate (5)
#                      +- * Expand (4)
#                         +- EventTimeWatermark (3)
#                            +- * Project (2)
#                               +- Scan csv  (1)

26/04/02 09:34:42 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8f5fbf76-3e9a-4404-bf4c-c3e8fd521108. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/02 09:34:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/02 09:34:42 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/04/02 09:35:02 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 7607 milliseconds


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+-------------+------------+
|              window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:34...|     Brooklyn|         122|
|{2026-04-02 09:34...|     Brooklyn|         122|
|{2026-04-02 09:34...|          N/A|           2|
|{2026-04-02 09:34...|    Manhattan|         138|
|{2026-04-02 09:34...|Staten Island|          40|
|{2026-04-02 09:34...|      Unknown|           2|
|{2026-04-02 09:34...|Staten Island|          40|
|{2026-04-02 09:34...|       Queens|         138|
|{2026-04-02 09:34...|          EWR|           2|
|{2026-04-02 09:34...|        Bronx|          86|
|{2026-04-02 09:34...|        Bronx|          86|
|{2026-04-02 09:34...|          N/A|           2|
|{2026-04-02 09:34...|          EWR|           2|
|{2026-04-02 09:34...|       Queens|         138|
|{2026-04-02 09:34...|    Manhattan|         138|
|{2

26/04/02 09:40:56 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6198 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-------------+------------+
|              window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:34...|     Brooklyn|         122|
|{2026-04-02 09:40...|     Brooklyn|         122|
|{2026-04-02 09:34...|     Brooklyn|         122|
|{2026-04-02 09:34...|          N/A|           2|
|{2026-04-02 09:40...|Staten Island|          40|
|{2026-04-02 09:34...|    Manhattan|         138|
|{2026-04-02 09:34...|Staten Island|          40|
|{2026-04-02 09:34...|      Unknown|           2|
|{2026-04-02 09:40...|      Unknown|           2|
|{2026-04-02 09:40...|       Queens|         138|
|{2026-04-02 09:34...|Staten Island|          40|
|{2026-04-02 09:34...|       Queens|         138|
|{2026-04-02 09:34...|          EWR|           2|
|{2026-04-02 09:40...|          N/A|           2|
|{2026-04-02 09:40...|    Manhattan|         138|
|{2

-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+-------------+------------+
|              window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:34...|     Brooklyn|         122|
|{2026-04-02 09:41...|          EWR|           2|
|{2026-04-02 09:40...|     Brooklyn|         244|
|{2026-04-02 09:34...|     Brooklyn|         122|
|{2026-04-02 09:34...|          N/A|           2|
|{2026-04-02 09:40...|Staten Island|          40|
|{2026-04-02 09:41...|          N/A|           2|
|{2026-04-02 09:34...|    Manhattan|         138|
|{2026-04-02 09:34...|Staten Island|          40|
|{2026-04-02 09:41...|    Manhattan|         138|
|{2026-04-02 09:40...|      Unknown|           2|
|{2026-04-02 09:34...|      Unknown|           2|
|{2026-04-02 09:40...|       Queens|         138|
|{2026-04-02 09:34...|Staten Island|          40|
|{2026-04-02 09:34...|       Queens|         138|
|{2

26/04/02 09:41:10 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5717 milliseconds


In [11]:
# session Window
from pyspark.sql.functions import col, from_json
import time
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import window, count,session_window

df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

processed = df.withColumn("event_time", current_timestamp())

# window(col("timestamp"), "10 minutes", "5 minutes") Window size: 10 mins, Slide: 5 mins 
windowed_df = processed \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(session_window('event_time','1 minutes') ,"Borough") \
    .agg(count("*").alias("record_count"))

query = windowed_df.writeStream \
    .format("console") \
    .outputMode("complete") \
    .trigger(processingTime="5 seconds") \
    .start()

# query.awaitTermination()

# == Physical Plan ==
# WriteToDataSourceV2 (12)
# +- * HashAggregate (11)
#    +- SessionWindowStateStoreSave (10)
#       +- MergingSessions (9)
#          +- SessionWindowStateStoreRestore (8)
#             +- * Sort (7)
#                +- Exchange (6)
#                   +- * HashAggregate (5)
#                      +- * Project (4)
#                         +- EventTimeWatermark (3)
#                            +- * Project (2)
#                               +- Scan csv  (1)

26/04/02 09:42:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-5a035b79-9c29-446c-9e7f-81d32906d7e3. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/02 09:42:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/02 09:42:17 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/04/02 09:42:35 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5937 milliseconds


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+-------------+------------+
|      session_window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:42...|       Queens|         138|
|{2026-04-02 09:42...|          EWR|           2|
|{2026-04-02 09:42...|      Unknown|           2|
|{2026-04-02 09:42...|     Brooklyn|         122|
|{2026-04-02 09:42...|Staten Island|          40|
|{2026-04-02 09:42...|          N/A|           2|
|{2026-04-02 09:42...|    Manhattan|         138|
|{2026-04-02 09:42...|        Bronx|          86|
+--------------------+-------------+------------+



26/04/02 09:42:51 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6447 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-------------+------------+
|      session_window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:42...|       Queens|         276|
|{2026-04-02 09:42...|          EWR|           4|
|{2026-04-02 09:42...|      Unknown|           4|
|{2026-04-02 09:42...|     Brooklyn|         244|
|{2026-04-02 09:42...|Staten Island|          80|
|{2026-04-02 09:42...|          N/A|           4|
|{2026-04-02 09:42...|    Manhattan|         276|
|{2026-04-02 09:42...|        Bronx|         172|
+--------------------+-------------+------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+-------------+------------+
|      session_window|      Borough|record_count|
+--------------------+-------------+------------+
|{2026-04-02 09:42...|       Queens|         276|
|{2026-04-02 09:46...|       Queens|         138|
|{2026-04-02 09:46...|          EWR|           2|
|{2026-04-02 09:42...|          EWR|           4|
|{2026-04-02 09:42...|      Unknown|           4|
|{2026-04-02 09:46...|      Unknown|           2|
|{2026-04-02 09:46...|     Brooklyn|         122|
|{2026-04-02 09:42...|     Brooklyn|         244|
|{2026-04-02 09:42...|Staten Island|          80|
|{2026-04-02 09:46...|Staten Island|          40|
|{2026-04-02 09:46...|          N/A|           2|
|{2026-04-02 09:42...|          N/A|           4|
|{2026-04-02 09:42...|    Manhattan|         276|
|{2026-04-02 09:46...|    Manhattan|         138|
|{2026-04-02 09:42...|        Bronx|         172|
|{2

26/04/02 09:46:06 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6706 milliseconds


In [11]:
query.stop()

26/04/03 15:17:52 WARN DAGScheduler: Failed to cancel job group 672412c1-fa40-4d4a-aba3-8461db9c5a1f. Cannot find active jobs for it.
26/04/03 15:17:52 WARN DAGScheduler: Failed to cancel job group 672412c1-fa40-4d4a-aba3-8461db9c5a1f. Cannot find active jobs for it.


# Output Modes
| Transformation | Append | Complete | Update |
|---|---|---|---|
| Filter / select / withColumn | ✅ | ❌ | ✅ |
| groupBy + count/sum | ❌ without watermark | ✅ | ✅ |
| groupBy + count/sum + watermark | ✅ finalized windows only | ✅ | ✅ |
| joins (stream-stream) | ✅ with watermark | ❌ | ❌ |
| joins (stream-static) | ✅ | ❌ | ✅ |
| dropDuplicates | ✅ with watermark | ❌ | ❌ |

### Append Mode

```
Complete Mode — emits EVERY batch, window open or not
Batch 0 (9:00)              Batch 1 (9:05)            Batch 2 (9:15)
+──────────────+──────+     +──────────────+──────+    +──────────────+──────+
│window        │count │     │window        │count │    │window        │count │
+──────────────+──────+     +──────────────+──────+    +──────────────+──────+
│09:00 - 09:05 │  3   │     │09:00 - 09:05 │  7   │   │09:00 - 09:05 │  9   │
+──────────────+──────+     │09:05 - 09:10 │  2   │   │09:05 - 09:10 │  5   │
                            +──────────────+──────+    │09:10 - 09:15 │  1   │
                                                       +──────────────+──────+
  ↑ partial window          ↑ still partial            ↑ still partial
  emitted immediately       emitted every batch        emitted every batch

Append Mode — emits ONLY when window is 100% finalized
Batch 0 (9:00)    Batch 1 (9:05)    Batch 2 (9:15)         Batch 3 (9:25)
                                                          +──────────────+──────+
  (nothing)         (nothing)          (nothing)          │09:00 - 09:05 │  9   │
                                                          +──────────────+──────+
  ↑ window not      ↑ window not       ↑ watermark          ↑ watermark passed!
  finalized yet     finalized yet      not passed yet        window is FROZEN
                                                            emitted ONCE, FINAL

In [ ]:
# append mode with stateless Transformation
from pyspark.sql.functions import col, from_json
import time

df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

df=df.filter(col('Borough')!='Queens')

query = df.writeStream \
    .outputMode("append")\
    .format("csv") \
    .option("path", "/app/stream_output/") \
    .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/") \
    .option("header", "true") \
    .trigger(processingTime="5 seconds") \
    .start()

query.awaitTermination()

26/04/02 09:12:20 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [ ]:
# Append mode with statefull Transformation
# In statefull transformation we use append mode but it not show or store middle level micro batches data , it shoow final calculated output across all batches that comes in single 
# watermark timestamp ,and thats why watermark timestamp is necessary to freeze the record and store 

from pyspark.sql.functions import col, from_json
import time

df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

processed = df.withColumn("event_time", current_timestamp())

windowed_df = processed \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(window("event_time", "2 minutes"),"Borough") \
    .agg(count("*").alias("record_count"))

query = df.writeStream \
    .outputMode("append")\
    .format("csv") \
    .option("path", "/app/stream_output/") \
    .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/") \
    .option("header", "true") \
    .trigger(processingTime="5 seconds") \
    .start()

query.awaitTermination()

## Complete Mode

Assume words arriving in CSV files:
- **Batch 0 input:** `hello, hello, world`
- **Batch 1 input:** `hello, spark, spark`
- **Batch 2 input:** `world, world`
---

#### COMPLETE MODE — Full table every batch
```
==================================================
  BATCH 0 — 3 rows written
==================================================
+-------+-----+
|word   |count|
+-------+-----+
|hello  |  2  |
|world  |  1  |
+-------+-----+

==================================================
  BATCH 1 — 3 rows written (FULL TABLE AGAIN)
==================================================
+-------+-----+
|word   |count|
+-------+-----+
|hello  |  3  |   ← updated (was 2, now 3)
|world  |  1  |   ← unchanged BUT still written
|spark  |  2  |   ← new
+-------+-----+

==================================================
  BATCH 2 — 3 rows written (FULL TABLE AGAIN)
==================================================
+-------+-----+
|word   |count|
+-------+-----+
|hello  |  3  |   ← unchanged BUT still written
|world  |  3  |   ← updated (was 1, now 3)
|spark  |  2  |   ← unchanged BUT still written
+-------+-----+


In [12]:
# Complete Mode with Tubling Window 
# csv file only support append mode

from pyspark.sql.functions import col, from_json
import time,os
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import window, count

df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

processed = df.withColumn("event_time", current_timestamp())

windowed_df = processed \
    .groupBy(window("event_time", "2 minutes"),"Borough") \
    .agg(count("*").alias("record_count"))

import os

# Use /app/ or /tmp/ — these are writable in Bitnami Spark containers
OUTPUT_PATH = "/app/output/results"
os.makedirs(OUTPUT_PATH, exist_ok=True)   # create it upfront from Python

def process_batch(batch_df, batch_id):

    flat_df = batch_df.withColumn("window_start", col("window.start")) \
                      .withColumn("window_end",   col("window.end")) \
                      .drop("window")

    flat_df.cache()

    print(f"\n{'='*50}")
    print(f"  BATCH: {batch_id}  |  Rows: {flat_df.count()}")
    print(f"{'='*50}")
    flat_df.show(truncate=False)

    flat_df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(f"/app/output/results/batch_{batch_id}")

    flat_df.unpersist()

query = windowed_df.writeStream \
    .outputMode("complete") \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/") \
    .trigger(processingTime="5 seconds") \
    .start()

# query.awaitTermination()

#In Meory table svan is important (peending)
# Must run and explore with detail from UI

# 0th id dataframe/sql
# == Physical Plan ==
# * HashAggregate (9)
# +- StateStoreSave (8)
#    +- * HashAggregate (7)
#       +- StateStoreRestore (6)
#          +- * HashAggregate (5)
#             +- Exchange (4)
#                +- * HashAggregate (3)
#                   +- * Project (2)
#                      +- Scan csv  (1)

# 1st id dataframe/sql
# == Physical Plan ==
# * HashAggregate (7)
# +- Exchange (6)
#    +- * HashAggregate (5)
#       +- InMemoryTableScan (1)
#             +- InMemoryRelation (2)
#                   +- * Project (4)
#                      +- * Scan ExistingRDD (3)

# 2nd id dataframe/sql
# == Physical Plan ==
# CollectLimit (6)
# +- * Project (5)
#    +- InMemoryTableScan (1)
#          +- InMemoryRelation (2)
#                +- * Project (4)
#                   +- * Scan ExistingRDD (3)

# 3rd id dataframe/sql
# == Physical Plan ==
# Execute InsertIntoHadoopFsRelationCommand (6)
# +- WriteFiles (5)
#    +- InMemoryTableScan (1)
#          +- InMemoryRelation (2)
#                +- * Project (4)
#                   +- * Scan ExistingRDD (3)

26/04/03 15:18:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/03 15:18:17 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


  BATCH: 0  |  Rows: 8
+-------------+------------+-------------------+-------------------+
|Borough      |record_count|window_start       |window_end         |
+-------------+------------+-------------------+-------------------+
|Staten Island|40          |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Queens       |138         |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Bronx        |86          |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Unknown      |2           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|N/A          |2           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Brooklyn     |122         |2026-04-03 15:18:00|2026-04-03 15:20:00|
|EWR          |2           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Manhattan    |138         |2026-04-03 15:18:00|2026-04-03 15:20:00|
+-------------+------------+-------------------+-------------------+



26/04/03 15:18:44 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 14408 milliseconds


  BATCH: 1  |  Rows: 8
+-------------+------------+-------------------+-------------------+
|Borough      |record_count|window_start       |window_end         |
+-------------+------------+-------------------+-------------------+
|Staten Island|80          |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Queens       |276         |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Bronx        |172         |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Unknown      |4           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|N/A          |4           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Brooklyn     |244         |2026-04-03 15:18:00|2026-04-03 15:20:00|
|EWR          |4           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Manhattan    |276         |2026-04-03 15:18:00|2026-04-03 15:20:00|
+-------------+------------+-------------------+-------------------+



26/04/03 15:19:11 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 16975 milliseconds


  BATCH: 2  |  Rows: 16
+-------------+------------+-------------------+-------------------+
|Borough      |record_count|window_start       |window_end         |
+-------------+------------+-------------------+-------------------+
|Manhattan    |138         |2026-04-03 15:20:00|2026-04-03 15:22:00|
|Staten Island|80          |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Queens       |276         |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Bronx        |86          |2026-04-03 15:20:00|2026-04-03 15:22:00|
|Bronx        |172         |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Unknown      |4           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Staten Island|40          |2026-04-03 15:20:00|2026-04-03 15:22:00|
|EWR          |2           |2026-04-03 15:20:00|2026-04-03 15:22:00|
|Brooklyn     |122         |2026-04-03 15:20:00|2026-04-03 15:22:00|
|N/A          |2           |2026-04-03 15:20:00|2026-04-03 15:22:00|
|N/A          |4           |2026-04-03 15:18:00|2026-04-03 15:20:00|
|Brooklyn 

26/04/03 15:21:22 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 12117 milliseconds


## UPDATE MODE — Only changed/new rows each batch
```
Assume words arriving in CSV files:
- **Batch 0 input:** `hello, hello, world`
- **Batch 1 input:** `hello, spark, spark`
- **Batch 2 input:** `world, world`
---
==================================================
  BATCH 0 — 2 rows written
==================================================
+-------+-----+
|word   |count|
+-------+-----+
|hello  |  2  |   ← new
|world  |  1  |   ← new
+-------+-----+

==================================================
  BATCH 1 — 2 rows written (ONLY CHANGES)
==================================================
+-------+-----+
|word   |count|
+-------+-----+
|hello  |  3  |   ← updated (changed so included)
|spark  |  2  |   ← new (included)
+-------+-----+
          ↑
          world NOT written — count didn't change!

==================================================
  BATCH 2 — 1 row written (ONLY CHANGES)
==================================================
+-------+-----+
|word   |count|
+-------+-----+
|world  |  3  |   ← updated (changed so included)
+-------+-----+
          ↑
          hello & spark NOT written — unchanged!


In [7]:
# Update Mode with Tubling Window 
# csv file only support append mode

from pyspark.sql.functions import col, from_json
import time,os
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import window, count

df = spark \
    .readStream \
    .format("csv") \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .load("/app/stream_csv/")

processed = df.withColumn("event_time", current_timestamp())

windowed_df = processed \
    .groupBy(window("event_time", "2 minutes"),"Borough") \
    .agg(count("*").alias("record_count"))

import os

# Use /app/ or /tmp/ — these are writable in Bitnami Spark containers
OUTPUT_PATH = "/app/output/results"
os.makedirs(OUTPUT_PATH, exist_ok=True)   # create it upfront from Python

def process_batch(batch_df, batch_id):

    flat_df = batch_df.withColumn("window_start", col("window.start")) \
                      .withColumn("window_end",   col("window.end")) \
                      .drop("window")

    flat_df.cache()

    print(f"\n{'='*50}")
    print(f"  BATCH: {batch_id}  |  Rows: {flat_df.count()}")
    print(f"{'='*50}")
    flat_df.show(truncate=False)

    flat_df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(f"/app/output/results/batch_{batch_id}")

    flat_df.unpersist()

query = windowed_df.writeStream \
    .outputMode("update") \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", f"/app/checkpoint_{int(time.time())}/") \
    .trigger(processingTime="5 seconds") \
    .start()

# query.awaitTermination()

26/04/03 15:36:24 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/03 15:36:24 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


  BATCH: 0  |  Rows: 8
+-------------+------------+-------------------+-------------------+
|Borough      |record_count|window_start       |window_end         |
+-------------+------------+-------------------+-------------------+
|EWR          |2           |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Brooklyn     |122         |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Bronx        |86          |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Unknown      |2           |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Manhattan    |138         |2026-04-03 15:36:00|2026-04-03 15:38:00|
|N/A          |2           |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Staten Island|40          |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Queens       |138         |2026-04-03 15:36:00|2026-04-03 15:38:00|
+-------------+------------+-------------------+-------------------+



26/04/03 15:36:53 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 18396 milliseconds


  BATCH: 1  |  Rows: 8
+-------------+------------+-------------------+-------------------+
|Borough      |record_count|window_start       |window_end         |
+-------------+------------+-------------------+-------------------+
|EWR          |4           |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Brooklyn     |244         |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Bronx        |172         |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Unknown      |4           |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Manhattan    |276         |2026-04-03 15:36:00|2026-04-03 15:38:00|
|N/A          |4           |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Staten Island|80          |2026-04-03 15:36:00|2026-04-03 15:38:00|
|Queens       |276         |2026-04-03 15:36:00|2026-04-03 15:38:00|
+-------------+------------+-------------------+-------------------+



26/04/03 15:37:38 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 13125 milliseconds


  BATCH: 2  |  Rows: 8
+-------------+------------+-------------------+-------------------+
|Borough      |record_count|window_start       |window_end         |
+-------------+------------+-------------------+-------------------+
|Queens       |138         |2026-04-03 15:38:00|2026-04-03 15:40:00|
|Unknown      |2           |2026-04-03 15:38:00|2026-04-03 15:40:00|
|EWR          |2           |2026-04-03 15:38:00|2026-04-03 15:40:00|
|Bronx        |86          |2026-04-03 15:38:00|2026-04-03 15:40:00|
|Manhattan    |138         |2026-04-03 15:38:00|2026-04-03 15:40:00|
|N/A          |2           |2026-04-03 15:38:00|2026-04-03 15:40:00|
|Brooklyn     |122         |2026-04-03 15:38:00|2026-04-03 15:40:00|
|Staten Island|40          |2026-04-03 15:38:00|2026-04-03 15:40:00|
+-------------+------------+-------------------+-------------------+



26/04/03 15:39:25 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 10983 milliseconds


In [9]:
query.stop()

26/04/03 15:39:48 WARN DAGScheduler: Failed to cancel job group 4961cab0-1291-42cc-8f0f-427f834877b4. Cannot find active jobs for it.
26/04/03 15:39:48 WARN DAGScheduler: Failed to cancel job group 4961cab0-1291-42cc-8f0f-427f834877b4. Cannot find active jobs for it.


### COMPLETE MODE — Real World
```
Business need: CEO dashboard showing total sales per category RIGHT NOW

Every 30 seconds, overwrite the dashboard data:

BATCH 0 (9:00 AM)              BATCH 1 (9:00:30 AM)
+-------------+--------+        +-------------+--------+
|category     |revenue |        |category     |revenue |
+-------------+--------+        +-------------+--------+
|Electronics  |$12,400 |        |Electronics  |$18,900 |  ← updated
|Clothing     |$8,200  |        |Clothing     |$9,100  |  ← updated
|Books        |$1,100  |        |Books        |$1,100  |  ← same but still sent
+-------------+--------+        +-------------+--------+
         ↑ full table sent every time — dashboard always correct

Business need: Top players scoreboard — must always show all players ranked

BATCH 3                         BATCH 4
+--------+-------+----+         +--------+-------+----+
|rank    |player |score         |rank    |player |score
+--------+-------+----+         +--------+-------+----+
|1       |Alice  |980 |         |1       |Bob    |1050|  ← rank changed
|2       |Bob    |950 |         |2       |Alice  |980 |  ← rank changed
|3       |Carol  |800 |         |3       |Carol  |800 |  ← same but needed
+--------+-------+----+         +--------+-------+----+
         ↑ if Carol's row not sent, leaderboard breaks
```

> Complete is **mandatory** here — if you use Update, unchanged players disappear from the board entirely.

### Update Mode
```
Bank Account Balance Sync
Business need: Update customer balances in the main DB as transactions arrive

BATCH 0                    BATCH 1 (only changed accounts)
+----------+--------+      +----------+--------+
|account_id|balance |      |account_id|balance |
+----------+--------+      +----------+--------+
|ACC001    |$5,000  |      |ACC001    |$4,200  |  ← withdrawal happened
|ACC002    |$12,000 |      |ACC003    |$8,500  |  ← deposit happened
|ACC003    |$8,000  |      +----------+--------+
+----------+--------+      ACC002 NOT sent — no activity

Business need: Update driver location in Redis/DB only when position changes

100,000 drivers on road. Each batch: maybe 500 moved.

COMPLETE would write:   100,000 rows every 10 seconds  ❌ DB killer
UPDATE writes:              500 rows every 10 seconds  ✅ efficient

+----------+---------+---------+
|driver_id |lat      |lon      |
+----------+---------+---------+
|DRV_0042  |31.5204  |74.3587  |  ← moved
|DRV_0891  |31.4678  |74.2901  |  ← moved
|DRV_2341  |30.1234  |71.4567  |  ← moved
+----------+---------+---------+
99,997 drivers not written — they didn't move ✅
```

## Production Decision Framework
```
Is the consumer a DASHBOARD or REPORT?
→ needs full picture always?
        YES → COMPLETE

Does the consumer need to REACT TO CHANGES only?
→ database sync, alerts, notifications?
        YES → UPDATE

Is your state table LARGE (millions of rows)?
→ writing all rows every batch too expensive?
        YES → UPDATE

Are unchanged rows MEANINGFUL to the consumer?
→ leaderboard positions, full rankings?
        YES → COMPLETE
        NO  → UPDATE

## Real Production Sources & Sinks by Output Mode
```
Sources (readStream) in Production
Socket/CSV ──→ Only for learning & testing
Kafka      ──→ 90% of real production streaming
Files(S3)  ──→ batch-like streaming, data lake ingestion  
Kinesis    ──→ AWS production systems
Delta Lake ──→ Lakehouse architecture

### COMPLETE MODE — Production Sources & Sinks

#### Typical Flow
```
Kafka (source)
    │
    ▼
Spark Structured Streaming
    │  outputMode("complete")
    ▼
Dashboard DB / Cache / BI Tool (sink)

### UPDATE MODE — Production Sources & Sinks

#### Typical Flow
```
Kafka (source)
    │
    ▼
Spark Structured Streaming
    │  outputMode("update")
    ▼
Kafka (sink) / Cassandra / MongoDB / Redis (UPSERT)

```
                    ┌─────────────┬─────────────┬─────────────┐
                    │   APPEND    │  COMPLETE   │   UPDATE    │
┌───────────────────┼─────────────┼─────────────┼─────────────┤
│ SOURCE            │             │             │             │
│ Kafka             │     ✅      │     ✅      │     ✅      │
│ S3 / HDFS files   │     ✅      │     ✅      │     ✅      │
│ Kinesis           │     ✅      │     ✅      │     ✅      │
│ Delta Lake        │     ✅      │     ✅      │     ✅      │
│ Socket (dev only) │     ✅      │     ✅      │     ✅      │
├───────────────────┼─────────────┼─────────────┼─────────────┤
│ SINK              │             │             │             │
│ Kafka             │     ✅      │     ✅      │     ✅      │
│ PostgreSQL/MySQL  │     ✅      │     ✅      │     ✅      │
│ Cassandra         │     ✅      │     ⚠️      │     ✅      │
│ MongoDB           │     ✅      │     ✅      │     ✅      │
│ Redis             │     ✅      │     ✅      │     ✅      │
│ Delta Lake        │     ✅      │     ❌      │     ❌      │
│ S3 / HDFS files   │     ✅      │     ❌      │     ❌      │
│ CSV files         │     ✅      │     ❌      │     ❌      │
│ Console           │     ✅      │     ✅      │     ✅      │
│ Memory            │     ✅      │     ✅      │     ✅      │
└───────────────────┴─────────────┴─────────────┴─────────────┘
⚠️  = works but not recommended   ❌ = not supported

In [6]:
# autoloader is databricks feaature 

from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

input_path = "/app/stream_csv/"
checkpoint_path = "/app/checkpoint/"


df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(schema)
    .load(input_path)
)

query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="30 seconds")
    .start("/app/output/")
)

Py4JJavaError: An error occurred while calling o60.load.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: cloudFiles. Make sure the provider name is correct and the package is properly registered and compatible with your Spark version. SQLSTATE: 42K02
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:764)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:686)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:75)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:45)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsUpWithPruning$3(AnalysisHelper.scala:139)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:107)
	at org.apache.spark.sql.catalyst.p

26/04/21 15:54:54 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 1086504 ms exceeds timeout 120000 ms
26/04/21 15:54:54 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 1083656 ms exceeds timeout 120000 ms
26/04/21 15:54:54 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.2: worker lost: Not receiving heartbeat for 60 seconds
26/04/21 15:54:55 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/04/21 15:54:55 WARN StandaloneSchedulerBackend: Executor to kill 1 does not exist!
